# TOKATÇI — Value v2 (kalibrasyonlu)

Değişiklikler:

- val seti **model-selection** ve **calibration** olarak ikiye ayrılır (episode bazlı),
- model seçimi **val logloss** ile yapılır,
- patience=10,
- en iyi modelden sonra **temperature scaling** yapılır,
- `temperature` model dosyasına kaydedilir.

Search tarafında:

```text
p = sigmoid(logit / temperature)
```


In [ ]:
from pathlib import Path
import numpy as np, torch
from torch import nn, optim

INPUT=Path('/kaggle/input')
DATA=next(INPUT.rglob('dataset_tokatci_b1.npz'),None)
if DATA is None: raise FileNotFoundError('dataset_tokatci_b1.npz yok')
OUT=Path('/kaggle/working/model_tokatci_value_v2.npz')
print(DATA, torch.__version__, 'cuda',torch.cuda.is_available())


In [ ]:
d=np.load(DATA)
state=d['state'].astype(np.float32)
y=d['game_win'].astype(np.float32)
w=d['sample_weight'].astype(np.float32)
split=d['split'].astype(np.int8)
ep=d['episode_id']

train_idx=np.nonzero(split==0)[0]
val_all=np.nonzero(split==1)[0]
# Episode bazlı calibration ayırımı: val episode'larının yarısı calibration.
val_eps=np.unique(ep[val_all]); rng=np.random.default_rng(123)
rng.shuffle(val_eps); ncal=max(1,len(val_eps)//2); cal_eps=set(map(int,val_eps[:ncal]))
cal_idx=np.array([i for i in val_all if int(ep[i]) in cal_eps],dtype=np.int64)
val_idx=np.array([i for i in val_all if int(ep[i]) not in cal_eps],dtype=np.int64)
print('rows',len(state),'train',len(train_idx),'val',len(val_idx),'cal',len(cal_idx),'cal_eps',ncal)

mean=state[train_idx].mean(0); std=state[train_idx].std(0)
weak=std<1e-4; mean[weak]=0; std[weak]=1


In [ ]:
class ValueNet(nn.Module):
 def __init__(self,dim=50,dropout=.1):
  super().__init__(); self.net=nn.Sequential(nn.Linear(dim,128),nn.ReLU(),nn.Dropout(dropout),nn.Linear(128,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,1))
 def forward(self,x): return self.net(x).squeeze(-1)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=ValueNet(state.shape[1]).to(device)
opt=optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-4)
loss_fn=nn.BCEWithLogitsLoss(reduction='none')
print('device',device,'params',sum(p.numel() for p in net.parameters()))


In [ ]:
def infer_logits(net,idx,chunk=8192):
 net.eval(); out=[]
 with torch.no_grad():
  for s in range(0,len(idx),chunk):
   ii=idx[s:s+chunk]
   xb=torch.from_numpy((state[ii]-mean)/std).to(device,non_blocking=True)
   out.append(net(xb).cpu().numpy())
 return np.concatenate(out)
def metrics(yy,logit):
 p=1/(1+np.exp(-logit))
 bce=float(np.mean(np.logaddexp(0,logit)-yy*logit))
 brier=float(np.mean((p-yy)**2)); acc=float(((p>=.5)==yy).mean())
 # AUC rank formula
 order=np.argsort(p); ranks=np.empty_like(order,dtype=np.float64); ranks[order]=np.arange(1,len(p)+1)
 n1=yy.sum(); n0=len(yy)-n1
 auc=float((ranks[yy==1].sum()-n1*(n1+1)/2)/(n1*n0)) if n1>0 and n0>0 else 0.5
 return bce,acc,auc,brier


In [ ]:
best=(-1,None,0); bad=0; patience=10; bs=1024
for epoch in range(1,41):
 net.train(); order=train_idx.copy(); np.random.shuffle(order)
 for s in range(0,len(order),bs):
  ii=order[s:s+bs]
  xb=torch.from_numpy((state[ii]-mean)/std).to(device,non_blocking=True)
  yb=torch.from_numpy(y[ii]).to(device,non_blocking=True)
  wb=torch.from_numpy(w[ii]).to(device,non_blocking=True)
  opt.zero_grad(set_to_none=True)
  loss=(loss_fn(net(xb),yb)*wb).sum()/wb.sum().clamp_min(1e-8)
  loss.backward(); opt.step()
 logits=infer_logits(net,val_idx); val=metrics(y[val_idx],logits)
 print(f'epoch {epoch:2d} val bce={val[0]:.4f} acc={val[1]:.4f} auc={val[2]:.4f} brier={val[3]:.4f}')
 # Model seçimi: logloss (kalibrasyona daha uygun)
 if val[0] < best[0] or best[0] < 0:
  best=(val[0],{k:v.detach().cpu().clone() for k,v in net.state_dict().items()},epoch); bad=0
 else:
  bad+=1
  if bad>=patience: print('early stop'); break
net.load_state_dict(best[1]); print('best',best[2])


In [ ]:
# Temperature scaling: sadece calibration set üzerinde.
cal_logits=infer_logits(net,cal_idx); ycal=y[cal_idx]
ts=np.linspace(0.5,3.0,501); losses=[]
for t in ts:
 losses.append(np.mean(np.logaddexp(0,cal_logits/t)-ycal*(cal_logits/t)))
T=float(ts[int(np.argmin(losses))])
print('temperature',T)
val_logits=infer_logits(net,val_idx)
print('val raw',metrics(y[val_idx],val_logits))
print('val cal',metrics(y[val_idx],val_logits/T))


In [ ]:
arrays={k:v.detach().cpu().numpy() for k,v in net.state_dict().items()}
arrays['state_mean']=mean.astype(np.float32); arrays['state_std']=std.astype(np.float32)
arrays['temperature']=np.array([T],dtype=np.float32)
np.savez_compressed(OUT,**arrays)
print('wrote',OUT,OUT.stat().st_size/1024,'KB')
